In [13]:
import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
import os

from tensorflow.keras.models import load_model

print("OpenCV :", cv2.__version__)
print("MediaPipe :", mp.__version__)
print("TensorFlow :", tf.__version__)

print("All Libraries Loaded Successfully!")

OpenCV : 4.10.0
MediaPipe : 0.10.35
TensorFlow : 2.21.0
All Libraries Loaded Successfully!


In [14]:
# Emotion Recognition Model
emotion_model = load_model("models/emotion_model.keras")

# Face Recognition Model
face_model = load_model("models/face_recognition_model.keras")

print("Models Loaded Successfully!")

ValueError: File not found: filepath=models/emotion_model.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
import os

print(os.listdir("models"))

In [ ]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if "emotion" in file.lower() and file.endswith(".keras"):
            print(os.path.join(root, file))

In [ ]:
from tensorflow.keras.models import load_model

# Emotion Model
emotion_model = load_model("models/emotion_recognition/emotion_model.keras")

# Face Recognition Model
face_model = load_model("models/face_recognition_model.keras")

print("Models Loaded Successfully!")

In [ ]:
face_detector = cv2.CascadeClassifier(
    "haarcascade_frontalface_default.xml"
)

print("Face Detector Ready!")

In [ ]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="face_landmarker.task"
    ),
    running_mode=RunningMode.VIDEO,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

print("Face Landmarker Ready!")

In [ ]:
cap = cv2.VideoCapture(0)

print(cap.isOpened())

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:

        face = frame[y:y+h, x:x+w]

        face = cv2.resize(face, (224, 224))
        face = face.astype("float32") / 255.0
        face = np.expand_dims(face, axis=0)

        prediction = face_model.predict(face, verbose=0)

        person = np.argmax(prediction)

        if person == 0:
            name = "Disha"
        else:
            name = "Unknown"

        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        cv2.putText(
            frame,
            name,
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,0),
            2
        )

    cv2.imshow("Student Engagement System", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
emotion_labels = [
    "Angry",
    "Disgust",
    "Fear",
    "Happy",
    "Neutral",
    "Sad",
    "Surprise"
]

print(emotion_labels)

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)


while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:

        # ---------- Face Recognition ----------
        face_rgb = frame[y:y+h, x:x+w]

        face_rgb = cv2.resize(face_rgb, (224, 224))
        face_rgb = face_rgb.astype("float32") / 255.0
        face_rgb = np.expand_dims(face_rgb, axis=0)

        face_pred = face_model.predict(face_rgb, verbose=0)
        name = "Disha"

        # ---------- Emotion ----------
        emotion_face = frame[y:y+h, x:x+w]

        emotion_face = cv2.resize(emotion_face, (224, 224))
        emotion_face = emotion_face.astype("float32") / 255.0
        emotion_face = np.expand_dims(emotion_face, axis=0)

        emotion_pred = emotion_model.predict(emotion_face, verbose=0)

        emotion = emotion_labels[np.argmax(emotion_pred)]

        # ---------- Display ----------
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        cv2.putText(
            frame,
            f"Name: {name}",
            (x, y-35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,255,0),
            2
        )

        cv2.putText(
            frame,
            f"Emotion: {emotion}",
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,0,0),
            2
        )

    cv2.imshow("Student Engagement", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import numpy as np
import mediapipe as mp

cap = cv2.VideoCapture(0)

EAR_THRESHOLD = 0.22
CONSEC_FRAMES = 3
blink_counter = 0
blink_total = 0

frame_number = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = landmarker.detect_for_video(mp_image, frame_number)

    # ---------------- Blink Detection ----------------
    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        left_eye = [(landmarks[i].x, landmarks[i].y) for i in LEFT_EYE]
        right_eye = [(landmarks[i].x, landmarks[i].y) for i in RIGHT_EYE]

        leftEAR = eye_aspect_ratio(left_eye)
        rightEAR = eye_aspect_ratio(right_eye)

        ear = (leftEAR + rightEAR) / 2.0

        cv2.putText(
            frame,
            f"EAR: {ear:.2f}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 0),
            2
            )

        if ear < EAR_THRESHOLD:
            blink_counter += 1
        elif blink_counter >= CONSEC_FRAMES:
            blink_total += 1
            blink_counter = 0
        else:
            blink_counter = 0

        cv2.putText(
            frame,
            f"Blinks: {blink_total}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

    # ---------------- Face Recognition + Emotion ----------------
    for (x, y, w, h) in faces:

        # Face Recognition
        face_rgb = frame[y:y+h, x:x+w]
        face_rgb = cv2.resize(face_rgb, (224, 224))
        face_rgb = face_rgb.astype("float32") / 255.0
        face_rgb = np.expand_dims(face_rgb, axis=0)

        face_pred = face_model.predict(face_rgb, verbose=0)

        # Since only one person is trained
        name = "Disha"

        # Emotion Recognition
        emotion_face = frame[y:y+h, x:x+w]
        emotion_face = cv2.resize(emotion_face, (224, 224))
        emotion_face = emotion_face.astype("float32") / 255.0
        emotion_face = np.expand_dims(emotion_face, axis=0)

        emotion_pred = emotion_model.predict(emotion_face, verbose=0)
        emotion = emotion_labels[np.argmax(emotion_pred)]

        # Draw Face Box
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0,255,0),
            2
        )

        # Display Name
        cv2.putText(
            frame,
            f"Name: {name}",
            (x, y-35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,255,0),
            2
        )

        # Display Emotion
        cv2.putText(
            frame,
            f"Emotion: {emotion}",
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,0,0),
            2
        )

    frame_number += 1

    cv2.imshow("Student Engagement", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# MediaPipe eye landmark indices

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

print("Eye Landmark Indices Loaded!")

In [ ]:
from scipy.spatial import distance

def eye_aspect_ratio(eye):
    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])

    ear = (A + B) / (2.0 * C)
    return ear

print("EAR Function Ready!")

In [ ]:
options = FaceLandmarkerOptions(
    ...
)

landmarker = FaceLandmarker.create_from_options(options)

print("Face Landmarker Ready!")

In [ ]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

options = FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="face_landmarker.task"
    ),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

print("Face Landmarker Ready!")

In [ ]:
import cv2

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

print(face_detector.empty())   # Should print False

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import time

cap = cv2.VideoCapture(0)

# ---------------- Blink Variables ----------------
EAR_THRESHOLD = 0.22
CONSEC_FRAMES = 3

blink_counter = 0
blink_total = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # ---------------- OpenCV Face Detection ----------------
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    # ---------------- MediaPipe Face Landmarks ----------------
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    # Real timestamp (fixes MediaPipe timestamp error)
    timestamp_ms = int(time.time() * 1000)

    result = landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    # ---------------- Blink Detection ----------------
    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        left_eye = [(landmarks[i].x, landmarks[i].y) for i in LEFT_EYE]
        right_eye = [(landmarks[i].x, landmarks[i].y) for i in RIGHT_EYE]

        leftEAR = eye_aspect_ratio(left_eye)
        rightEAR = eye_aspect_ratio(right_eye)

        ear = (leftEAR + rightEAR) / 2.0

        if ear < EAR_THRESHOLD:
            blink_counter += 1

        elif blink_counter >= CONSEC_FRAMES:
            blink_total += 1
            blink_counter = 0

        else:
            blink_counter = 0

        cv2.putText(
            frame,
            f"EAR: {ear:.2f}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,255,0),
            2
        )

        cv2.putText(
            frame,
            f"Blinks: {blink_total}",
            (20,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,255),
            2
        )

    # ---------------- Face Recognition + Emotion ----------------
    for (x, y, w, h) in faces:

        # Face Recognition
        face_rgb = frame[y:y+h, x:x+w]
        face_rgb = cv2.resize(face_rgb, (224,224))
        face_rgb = face_rgb.astype("float32") / 255.0
        face_rgb = np.expand_dims(face_rgb, axis=0)

        face_pred = face_model.predict(face_rgb, verbose=0)

        # Only one enrolled person
        name = "Disha"

        # Emotion Recognition
        emotion_face = frame[y:y+h, x:x+w]
        emotion_face = cv2.resize(emotion_face, (224,224))
        emotion_face = emotion_face.astype("float32") / 255.0
        emotion_face = np.expand_dims(emotion_face, axis=0)

        emotion_pred = emotion_model.predict(
            emotion_face,
            verbose=0
        )

        emotion = emotion_labels[np.argmax(emotion_pred)]

        # Face Box
        cv2.rectangle(
            frame,
            (x,y),
            (x+w,y+h),
            (0,255,0),
            2
        )

        # Name
        cv2.putText(
            frame,
            f"Name: {name}",
            (x,y-35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,255,0),
            2
        )

        # Emotion
        cv2.putText(
            frame,
            f"Emotion: {emotion}",
            (x,y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,0,0),
            2
        )

    cv2.imshow("Student Engagement", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="face_landmarker.task"
    ),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

print("Face Landmarker Ready!")

In [ ]:
print(landmarker)

In [ ]:
# MediaPipe eye landmark indices

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

print("Eye Landmark Indices Loaded!")

In [ ]:
from scipy.spatial import distance

def eye_aspect_ratio(eye):
    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])

    ear = (A + B) / (2.0 * C)
    return ear

print("EAR Function Ready!")

In [ ]:
print(face_detector)
print(face_model)
print(emotion_model)
print(landmarker)
print(LEFT_EYE)
print(RIGHT_EYE)
print(eye_aspect_ratio)

In [ ]:
from tensorflow.keras.models import load_model

face_model = load_model("models/face_recognition_model.keras")

print("Face Recognition Model Loaded!")

In [ ]:
from tensorflow.keras.models import load_model

emotion_model = load_model(
    "models/emotion_recognition/emotion_model.keras"
)

print("Emotion Model Loaded!")

In [ ]:
print(face_detector)
print(face_model)
print(emotion_model)
print(landmarker)
print(LEFT_EYE)
print(RIGHT_EYE)
print(eye_aspect_ratio)